# CAZ Validation — Paper 3 Companion

*Concept Encoding Strategies Across 26 Transformers: A Concept Allocation Zone Evaluation*

*Written: 2026-05-20 02:31 UTC*

This notebook accompanies Henry (2026c) — the validation paper for the CAZ Framework.
It reproduces or demonstrates every major quantitative claim from that paper using the
`paper_n250` dataset. A reader with access to that dataset can verify the paper results
using this notebook alone.

**What the validation paper establishes:**
1. A statistically robust **concept ordering tendency** across 26 base models (median per-model Kendall τ = 0.54, p < 0.001; 87% of models positively correlated)
2. **Scored detection** recovers 3.1× more structure than threshold detection (201 → 623 CAZes)
3. **Direction-specific suppression**: concept-direction ablation at CAZ peaks produces 3.67× more separation reduction than non-CAZ layers; random-direction ablation at the same layer produces near zero
4. **Architecture-conditioned causal roles**: MHA mean ablation effect 0.420 vs GQA 0.261 (p < 0.001)
5. **Phi-2 inversion**: sole outlier with τ = −0.25 (reversed ordering); synthetic-textbook training is the leading hypothesis

**Corpus**: 26 base models from 8 architecture families (70M–9B params), 7 semantic concepts.
No GPU required — all results use pre-computed extraction outputs.

**Related notebooks:**
- `01_caz_framework_introduction.ipynb` — CAZ concepts, signals, zone anatomy
- `03_caz_implementation_demo.ipynb` — PRH cross-architecture convergence (Paper 4)
- `04_gem_paper_companion.ipynb` — GEM framework validation (Paper 2)

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026a–d)

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    _pip("rosetta_tools>=1.3.1")
except subprocess.CalledProcessError:
    _pip("rosetta_tools @ git+https://github.com/jamesrahenry/Rosetta_Tools.git@v1.3.1")

_pip("huggingface_hub", "matplotlib", "numpy", "scipy")

In [ ]:
import json
import csv
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import kendalltau, mannwhitneyu
from huggingface_hub import hf_hub_download

# rosetta_tools: GPU host path first, then dev machine
_rt = Path.home() / "rosetta_tools"
if not _rt.exists():
    _rt = Path.home() / "Source" / "Rosetta_Program" / "rosetta_tools"
sys.path.insert(0, str(_rt))

from rosetta_tools.caz import LayerMetrics, find_caz_regions
from rosetta_tools.viz_style import concept_color, THEME, apply_theme

# Data paths
ROSETTA_DATA = Path.home() / "rosetta_data"
PAPER_N250   = ROSETTA_DATA / "paper_n250"
CAV_RESULTS  = ROSETTA_DATA / "results" / "CAZ_Validation"
RAND_RESULTS = ROSETTA_DATA / "results" / "random_control"

HF_REPO      = "james-ra-henry/Rosetta-Activations"
HF_DATA_ROOT = "paper_n250"

# The 7 Paper 3 concepts
P3_CONCEPTS = [
    "credibility", "negation", "causation",
    "temporal_order", "sentiment", "certainty", "moral_valence",
]

# Reference ordering from paper (Table 3, shallowest → deepest)
REFERENCE_ORDER = [
    "credibility", "negation", "causation",
    "temporal_order", "sentiment", "certainty", "moral_valence",
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "figure.facecolor": "white",
})
print("Setup complete.")

## Load pre-computed CAZ data

Pre-computed CAZ profiles for the `paper_n250` dataset are stored in the
[Rosetta Activations](https://huggingface.co/datasets/james-ra-henry/Rosetta-Activations)
HF dataset, or locally at `~/rosetta_data/paper_n250/`. Each file contains
layer-wise Fisher separation, coherence, velocity, and the dominant concept direction.

We first load two representative models used throughout this notebook:
- **Pythia-160M** — small MHA model, richest multimodal structure in dataset
- **Qwen2.5-1.5B** — GQA model, contrast cohort

In [ ]:
def model_dir_name(model_id: str) -> str:
    """Convert HF model ID to paper_n250 directory name."""
    return model_id.replace("/", "_").replace("-", "_")

def load_caz(model_id: str, concept: str) -> dict:
    """Load pre-computed CAZ JSON for a model/concept pair.
    
    Tries local paper_n250 first; falls back to HF download.
    """
    local_dir = PAPER_N250 / model_dir_name(model_id)
    local_path = local_dir / f"caz_{concept}.json"
    if local_path.exists():
        with open(local_path) as f:
            return json.load(f)
    # Fall back to HF
    filename = f"{HF_DATA_ROOT}/{model_id.replace('/', '_')}/caz_{concept}.json"
    path = hf_hub_download(HF_REPO, filename=filename, repo_type="dataset")
    with open(path) as f:
        return json.load(f)

def metrics_from_caz(data: dict) -> list[LayerMetrics]:
    """Convert loaded CAZ JSON metrics to LayerMetrics list."""
    return [
        LayerMetrics(
            layer=m["layer"],
            separation=m["separation_fisher"],
            coherence=m["coherence"],
            velocity=m.get("velocity", 0.0),
        )
        for m in data["layer_data"]["metrics"]
    ]

# Load two demo models
DEMO_MHA  = "EleutherAI/pythia-160m"
DEMO_GQA  = "Qwen/Qwen2.5-1.5B"

caz_mha  = {c: load_caz(DEMO_MHA, c) for c in P3_CONCEPTS}
caz_gqa  = {c: load_caz(DEMO_GQA, c) for c in P3_CONCEPTS}

n_mha = caz_mha[P3_CONCEPTS[0]]["n_layers"]
n_gqa = caz_gqa[P3_CONCEPTS[0]]["n_layers"]
print(f"Loaded Pythia-160M   ({n_mha}L MHA+GELU)")
print(f"Loaded Qwen2.5-1.5B  ({n_gqa}L GQA+SwiGLU)")
print(f"\n7 concepts: {', '.join(P3_CONCEPTS)}")

## 1. Concept Ordering

**Paper claim (§3.1):** Averaging CAZ peak depths across all 26 base models yields a consistent
ordering: credibility is shallowest, moral valence is deepest. Kendall τ between each model's
per-concept depth ranking and this mean ordering gives **median τ = 0.54** (mean τ = 0.45),
with 87% of models showing positive correlation (z = 11.5, p < 0.001 by permutation test).

**Reference ordering (Table 3):**

| Rank | Concept | Mean depth | Std |
|------|---------|-----------|-----|
| 1 (earliest) | credibility | 39.6% | 29.7 |
| 2 | negation | 49.8% | 17.9 |
| 3 | causation | 53.6% | 16.6 |
| 4 | temporal_order | 56.1% | 17.1 |
| 5 | sentiment | 61.8% | 13.6 |
| 6 | certainty | 64.2% | 14.2 |
| 7 (deepest) | moral_valence | 68.1% | 14.1 |

Here we demonstrate the τ computation on two representative models, then
load the pre-computed scored_analysis.csv to reproduce the aggregate result
across all 26 base models.

In [ ]:
def peak_depth_pct(data: dict) -> float:
    """Extract dominant (tallest) peak depth as % of model depth."""
    return data["layer_data"]["peak_depth_pct"]

def concept_tau(caz_dict: dict, ref_order: list[str]) -> tuple[float, float]:
    """Compute Kendall tau between a model's concept depth ranks and reference ordering.
    
    Returns (tau, p_value).
    """
    depths = {c: peak_depth_pct(caz_dict[c]) for c in ref_order if c in caz_dict}
    if len(depths) < 3:
        return float("nan"), float("nan")
    ordered_concepts = [c for c in ref_order if c in depths]
    observed_depths  = [depths[c] for c in ordered_concepts]
    ref_ranks        = list(range(len(ordered_concepts)))  # 0=shallowest in reference
    tau, p = kendalltau(ref_ranks, observed_depths)
    # Note: deeper depth = higher rank in reference → positive tau = same ordering
    # But reference ranks are ascending (0=shallowest), depths are ascending too,
    # so positive tau = agrees with reference.
    return tau, p

# Demo: single model τ computation
for model_id, caz_dict, label in [
    (DEMO_MHA, caz_mha, "Pythia-160M (MHA)"),
    (DEMO_GQA, caz_gqa, "Qwen2.5-1.5B (GQA)"),
]:
    depths = {c: peak_depth_pct(caz_dict[c]) for c in P3_CONCEPTS}
    tau, p = concept_tau(caz_dict, REFERENCE_ORDER)
    print(f"\n{label}:")
    for c in REFERENCE_ORDER:
        print(f"  {c:20s}  peak = {depths[c]:.1f}%")
    print(f"  → Kendall τ = {tau:.3f}  (p = {p:.4f})")

In [ ]:
# Reproduce aggregate τ statistic across all models in scored_analysis.csv
# Using the dominant peak per concept per model.

# Load scored_analysis
scored_csv = CAV_RESULTS / "scored_analysis.csv"
assert scored_csv.exists(), f"scored_analysis.csv not found at {scored_csv}"

# Build {model_id: {concept: dominant_peak_depth_pct}} from CSV
model_peaks = {}  # model_id -> concept -> depth_pct of dominant peak
with open(scored_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["concept"] not in P3_CONCEPTS:
            continue
        if row["is_dominant"] != "True":
            continue
        m = row["model_id"]
        c = row["concept"]
        model_peaks.setdefault(m, {})[c] = float(row["depth_pct"])

# Compute τ per model; keep only models with all 7 concepts
taus = []
model_tau_rows = []
for model_id, peaks in model_peaks.items():
    if len(peaks) < len(P3_CONCEPTS):
        continue
    observed = [peaks[c] for c in REFERENCE_ORDER]
    ref_ranks = list(range(len(REFERENCE_ORDER)))
    tau, p = kendalltau(ref_ranks, observed)
    if not np.isnan(tau):
        taus.append(tau)
        model_tau_rows.append((model_id, tau, p))

taus = np.array(taus)
n_positive = int((taus > 0).sum())
print(f"Models with all 7 P3 concepts: {len(taus)}")
print(f"Mean τ:   {taus.mean():.3f}")
print(f"Median τ: {np.median(taus):.3f}")
print(f"Positive τ: {n_positive}/{len(taus)} ({100*n_positive/len(taus):.1f}%)")
print()
print("Paper reports: mean τ = 0.45, median τ = 0.54, 87% positive, z = 11.5, p < 0.001")
print("NOTE: Paper uses 24 non-degenerate models (excludes OPT-125m floor effect + 2 instruct-only);")
print("      scored_analysis.csv includes more models — numbers may differ slightly.")

In [ ]:
# Visualise: per-model concept peak depths as a heatmap (reproduces Fig. 1 pattern)
# Use the subset of models that have all 7 concepts.

# Sort models by family (rough grouping by model_id prefix) then by τ within family
sorted_rows = sorted(model_tau_rows, key=lambda r: (r[0].split("/")[0], -r[1]))
model_ids  = [r[0] for r in sorted_rows]
depth_matrix = np.array(
    [[model_peaks[m][c] for c in REFERENCE_ORDER] for m in model_ids]
)

fig, ax = plt.subplots(figsize=(max(14, len(model_ids) * 0.35 + 2), 5))
fig.patch.set_facecolor("white")

im = ax.imshow(depth_matrix.T, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=100,
               origin="upper")
plt.colorbar(im, ax=ax, label="Peak depth (% of layers)", shrink=0.8)

ax.set_yticks(range(len(REFERENCE_ORDER)))
ax.set_yticklabels([c.replace("_", " ") for c in REFERENCE_ORDER], fontsize=9)
ax.set_xticks(range(len(model_ids)))
labels = [m.split("/")[-1][:14] for m in model_ids]
ax.set_xticklabels(labels, rotation=70, ha="right", fontsize=6)
ax.set_title(
    "CAZ peak depth — 7 concepts × models  (green=shallow, red=deep)\n"
    "Concept ordering tendency: cool-to-warm gradient top→bottom across most models",
    fontsize=11, fontweight="bold",
)
apply_theme(ax)
plt.tight_layout()
plt.show()

print("Reproduces Figure 1 from Henry (2026c). Phi-2 visible as an inversion (§1.2 below).")

In [ ]:
# Distribution of per-model τ values
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("white")

bins = np.linspace(-1, 1, 22)
ax.hist(taus, bins=bins, color="#1565C0", alpha=0.75, edgecolor="white")
ax.axvline(0, color="#C62828", lw=1.5, ls="--", label="τ = 0")
ax.axvline(float(np.median(taus)), color="#2E7D32", lw=2, ls="-",
           label=f"Median τ = {np.median(taus):.3f}")
ax.axvline(float(taus.mean()), color="#E65100", lw=2, ls=":",
           label=f"Mean τ = {taus.mean():.3f}")

ax.set_xlabel("Kendall τ vs reference ordering", fontsize=11)
ax.set_ylabel("Number of models", fontsize=11)
ax.set_title(
    f"Per-model concept ordering consistency  ({len(taus)} models with all 7 concepts)\n"
    f"{n_positive}/{len(taus)} ({100*n_positive/len(taus):.1f}%) show positive correlation",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9)
apply_theme(ax)
plt.tight_layout()
plt.show()

## 2. The Phi-2 Inversion

**Paper claim (§3.2):** Phi-2 (Microsoft, 2.8B, MHA+GELU, synthetic-textbook training)
is the sole clear outlier with **τ = −0.25** — its concept ordering is approximately
reversed. Affective and syntactic concepts assemble first (25–29% depth); epistemic
and relational concepts last (62–68%). This inversion holds across all 7 concepts,
ruling out measurement noise.

Phi-2 receives the same contrastive pairs as every other model but produces the
opposite ordering — evidence that ordering is an interaction between model and pairs,
not a property of the pairs alone.

In [ ]:
phi2_id = "microsoft/phi-2"
caz_phi2 = {c: load_caz(phi2_id, c) for c in P3_CONCEPTS}

# Compute tau for Phi-2
depths_phi2 = {c: peak_depth_pct(caz_phi2[c]) for c in P3_CONCEPTS}
observed_phi2 = [depths_phi2[c] for c in REFERENCE_ORDER]
ref_ranks = list(range(len(REFERENCE_ORDER)))
tau_phi2, p_phi2 = kendalltau(ref_ranks, observed_phi2)

# Compare Phi-2 against Pythia-2.8B (same architecture family, natural text)
pythia28_id = "EleutherAI/pythia-2.8b"
caz_p28 = {c: load_caz(pythia28_id, c) for c in P3_CONCEPTS}
depths_p28 = {c: peak_depth_pct(caz_p28[c]) for c in P3_CONCEPTS}
tau_p28, p_p28 = concept_tau(caz_p28, REFERENCE_ORDER)

print("Phi-2 vs Pythia-2.8b — same architecture family, different training data")
print(f"{'Concept':20s}  {'Phi-2 depth':>12}  {'Pythia-2.8B':>12}  {'Ref rank':>9}")
for i, c in enumerate(REFERENCE_ORDER):
    print(f"{c:20s}  {depths_phi2[c]:>11.1f}%  {depths_p28[c]:>11.1f}%  {i+1:>9}")
print()
print(f"Phi-2 τ = {tau_phi2:.3f} (p = {p_phi2:.4f})")
print(f"Pythia-2.8B τ = {tau_p28:.3f} (p = {p_p28:.4f})")
print()
print("Paper reports Phi-2 τ = −0.25. Architecture (33L, 2560d, MHA, RoPE, GELU)")
print("matches Pythia-2.8B closely — architecture is not the explanation.")

In [ ]:
# Side-by-side bar chart: Phi-2 vs Pythia-2.8B peak depths
x = np.arange(len(REFERENCE_ORDER))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4.5))
fig.patch.set_facecolor("white")

bars_phi = ax.bar(x - w/2, [depths_phi2[c] for c in REFERENCE_ORDER],
                  w, label=f"Phi-2  (τ={tau_phi2:.2f})",
                  color="#C62828", alpha=0.80, edgecolor="white")
bars_p28 = ax.bar(x + w/2, [depths_p28[c] for c in REFERENCE_ORDER],
                  w, label=f"Pythia-2.8B  (τ={tau_p28:.2f})",
                  color="#1565C0", alpha=0.80, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", " ") for c in REFERENCE_ORDER], fontsize=10)
ax.set_ylabel("Peak depth (% of model)", fontsize=10)
ax.set_ylim(0, 100)
ax.set_title(
    "Phi-2 inversion: affective concepts assembled first, epistemic last\n"
    "Pythia-2.8B (same architecture, natural text) follows the reference ordering",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9)
apply_theme(ax)
plt.tight_layout()
plt.show()

## 3. Scored Detection: 201 → 623 CAZes

**Paper claim (§2.4, §4.1):** The legacy threshold detector (10% prominence floor)
finds **201 CAZes** across 22 models. Scored detection (0.5% floor, composite
CAZ score = prominence × coherence_boost × √width) finds **623 CAZes** across
26 models — a **3.1× increase**.

| Category | Score | Count | % |
|----------|-------|-------|---|
| Major | > 0.5 | 103 | 17% |
| Strong | 0.2–0.5 | 86 | 14% |
| Moderate | 0.05–0.2 | 117 | 19% |
| Gentle | < 0.05 | 317 | 51% |

The 422 additional CAZes are predominantly gentle (score < 0.05) — causally
active in 93% of ablation tests (§6.1) despite being invisible to standard peak detection.

We demonstrate on Pythia-160M and Qwen2.5-1.5B, then report the aggregate numbers
from `scored_analysis.csv`.

In [ ]:
# Default (10%) vs scored (0.5%) detection — causation in both demo models

def score_category(score: float) -> str:
    if score > 0.5:  return "major"
    if score > 0.2:  return "strong"
    if score > 0.05: return "moderate"
    return "gentle"

DEMO_CONCEPT = "causation"
ZONE_COLORS  = ["#1565C0", "#E65100", "#2E7D32", "#7B1FA2", "#880E4F"]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.patch.set_facecolor("white")
fig.suptitle(
    f"Default (10% floor) vs scored (0.5% floor) detection — '{DEMO_CONCEPT}'",
    fontsize=12, fontweight="bold",
)

for row_idx, (model_id, caz_dict, label) in enumerate([
    (DEMO_MHA, caz_mha, "Pythia-160M (MHA)"),
    (DEMO_GQA, caz_gqa, "Qwen2.5-1.5B (GQA)"),
]):
    data     = caz_dict[DEMO_CONCEPT]
    metrics  = data["layer_data"]["metrics"]
    n_layers = data["n_layers"]
    depth_x  = [m["layer"] / n_layers * 100 for m in metrics]
    sep      = [m["separation_fisher"] for m in metrics]
    color    = concept_color(DEMO_CONCEPT)

    lm = metrics_from_caz(data)
    prof_default = find_caz_regions(lm)                               # 10% floor
    prof_scored  = find_caz_regions(lm, min_prominence_frac=0.005)   # 0.5% floor

    for col_idx, (prof, det_label) in enumerate([
        (prof_default, f"Default 10% — {prof_default.n_regions} zone(s)"),
        (prof_scored,  f"Scored 0.5% — {prof_scored.n_regions} zone(s)"),
    ]):
        ax = axes[row_idx][col_idx]
        ymax = max(sep) * 1.3
        ax.plot(depth_x, sep, color=color, lw=2.0, zorder=3)
        ax.fill_between(depth_x, sep, alpha=0.07, color=color)
        for zi, region in enumerate(prof.regions):
            s_pct = region.start / n_layers * 100
            e_pct = region.end   / n_layers * 100
            zc = ZONE_COLORS[zi % len(ZONE_COLORS)]
            ax.axvspan(s_pct, e_pct, alpha=0.18, color=zc)
            ax.axvline(region.depth_pct, color=zc, lw=1.4, ls="--", alpha=0.85)
            cat = score_category(region.caz_score)
            ax.text(region.depth_pct, ymax * 0.97,
                    f"Z{zi+1}\n{region.depth_pct:.0f}%\n({cat})",
                    ha="center", va="top", fontsize=7, color=zc,
                    fontweight="bold", clip_on=False)
        ax.set_title(f"{label}\n{det_label}", fontsize=9, fontweight="bold")
        ax.set_xlabel("Depth (%)", fontsize=9)
        ax.set_ylabel("S(l) — Separation", fontsize=9)
        ax.set_xlim(0, depth_x[-1])
        ax.set_ylim(0, ymax)
        apply_theme(ax)

plt.tight_layout()
plt.show()

# Print zone details
for model_id, caz_dict, label in [
    (DEMO_MHA, caz_mha, "Pythia-160M"),
    (DEMO_GQA, caz_gqa, "Qwen2.5-1.5B"),
]:
    lm = metrics_from_caz(caz_dict[DEMO_CONCEPT])
    prof = find_caz_regions(lm, min_prominence_frac=0.005)
    n_layers = caz_dict[DEMO_CONCEPT]["n_layers"]
    print(f"\n{label} — {DEMO_CONCEPT} — {prof.n_regions} scored zones:")
    for zi, r in enumerate(prof.regions):
        print(f"  Z{zi+1}: peak {r.depth_pct:.0f}%  score={r.caz_score:.3f}  ({score_category(r.caz_score)})")

In [ ]:
# Aggregate: reproduce the 201 → 623 count from scored_analysis.csv
# The CSV was generated with the 0.5% floor (scored); we replicate the 10% threshold
# count by filtering on caz_score >= threshold equivalent.

import csv as _csv

total_scored = 0
models_scored = set()
total_dominant = 0  # proxy for legacy single-peak count
score_counts = {"major": 0, "strong": 0, "moderate": 0, "gentle": 0}

with open(scored_csv) as f:
    reader = _csv.DictReader(f)
    for row in reader:
        if row["concept"] not in P3_CONCEPTS:
            continue
        total_scored += 1
        models_scored.add(row["model_id"])
        score = float(row["caz_score"])
        score_counts[score_category(score)] += 1
        if row["is_dominant"] == "True":
            total_dominant += 1

print("Scored detection (0.5% floor) — P3 concepts only:")
print(f"  Total CAZes detected: {total_scored}")
print(f"  Unique models:        {len(models_scored)}")
print(f"  Dominant peaks only:  {total_dominant}  (≈ legacy threshold count)")
print()
print("Score distribution:")
for cat in ["major", "strong", "moderate", "gentle"]:
    pct = 100 * score_counts[cat] / total_scored
    print(f"  {cat:10s}: {score_counts[cat]:4d}  ({pct:.0f}%)")
print()
print("Paper reports: 201 (threshold) → 623 (scored) = 3.1× increase")
print("(Paper uses all 17 concepts; P3 covers 7. Ratio should be consistent.)")
if total_dominant > 0:
    ratio = total_scored / total_dominant
    print(f"This subset: {total_dominant} → {total_scored} = {ratio:.1f}×")

## 4. Suppression Ratio: 3.67× at CAZ Peaks vs Non-CAZ Layers

**Paper claim (§6.1, Table 11):** Ablating the concept direction at **CAZ peak layers**
produces a mean separation reduction of **0.406** vs **0.111** at non-CAZ layers
(>3 layers from any detected peak). That is a **3.67× ratio** (Mann-Whitney
p = 6.01 × 10⁻³¹).

We reproduce this by loading the `ablation_global_sweep_*.json` files which contain
layer-by-layer ablation results. These require pre-computed ablation data — running
ablations requires GPU and hours of compute. The files are loaded from
`~/rosetta_data/paper_n250/<model>/ablation_global_sweep_<concept>.json`.

The cells below show the computation pattern on Pythia-160M and report the
aggregate across all models where data is available.

In [ ]:
def load_global_sweep(model_id: str, concept: str) -> dict | None:
    """Load ablation_global_sweep JSON. Returns None if not available locally."""
    path = PAPER_N250 / model_dir_name(model_id) / f"ablation_global_sweep_{concept}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# Demo: ablation sweep for causation in Pythia-160M
sweep = load_global_sweep(DEMO_MHA, "causation")

if sweep:
    caz_data = load_caz(DEMO_MHA, "causation")
    lm = metrics_from_caz(caz_data)
    prof = find_caz_regions(lm, min_prominence_frac=0.005)
    caz_peak_layers = {r.peak for r in prof.regions}

    n_layers  = sweep["n_layers"]
    layers    = sweep["layers"]
    depths    = [l["depth_pct"] for l in layers]
    reductions= [l["global_sep_reduction"] for l in layers]
    layer_idxs= [l["layer"] for l in layers]

    # Classify as CAZ or non-CAZ (>3 layers from any peak)
    is_caz = [any(abs(li - pk) <= 0 for pk in caz_peak_layers) for li in layer_idxs]
    is_noncaz = [all(abs(li - pk) > 3 for pk in caz_peak_layers) for li in layer_idxs]

    caz_reds    = [r for r, c in zip(reductions, is_caz) if c]
    noncaz_reds = [r for r, c in zip(reductions, is_noncaz) if c]

    print(f"Pythia-160M — causation ablation sweep")
    print(f"  CAZ peak layers:    {[li for li, c in zip(layer_idxs, is_caz) if c]}")
    print(f"  Mean sep. reduction at CAZ peaks:    {np.mean(caz_reds):.3f}")
    print(f"  Mean sep. reduction at non-CAZ:      {np.mean(noncaz_reds):.3f}")
    if noncaz_reds:
        print(f"  Ratio:                               {np.mean(caz_reds)/np.mean(noncaz_reds):.2f}×")

    # Plot
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor("white")
    colors_plot = ["#C62828" if c else ("#1565C0" if n else "#9E9E9E")
                   for c, n in zip(is_caz, is_noncaz)]
    ax.bar(depths, reductions, width=depths[1]-depths[0] if len(depths)>1 else 1,
           color=colors_plot, alpha=0.8)
    ax.set_xlabel("Depth (% of layers)", fontsize=10)
    ax.set_ylabel("Separation reduction", fontsize=10)
    ax.set_title("Ablation sweep — causation in Pythia-160M\n"
                 "Red = CAZ peak layers · Blue = non-CAZ layers",
                 fontsize=11, fontweight="bold")
    apply_theme(ax)
    plt.tight_layout()
    plt.show()
else:
    print("Ablation sweep data not available locally for Pythia-160M.")
    print("Ablation requires GPU compute — results are pre-computed.")
    print("Paper result: CAZ peaks 0.406, non-CAZ 0.111, ratio 3.67×")
    print("(p = 6.01 × 10⁻³¹, Mann-Whitney U = 29,799)")

In [ ]:
# Aggregate 3.67× ratio across all available models

all_caz_reds    = []
all_noncaz_reds = []

available_models = sorted(p.name for p in PAPER_N250.iterdir() if p.is_dir())
for model_dir_str in available_models:
    # Reconstruct model_id from directory name (reverse the _ → / and _ → - mapping)
    # We match against models known to have global sweeps by checking for files.
    for concept in P3_CONCEPTS:
        sweep = None
        path = PAPER_N250 / model_dir_str / f"ablation_global_sweep_{concept}.json"
        if path.exists():
            with open(path) as f:
                sweep = json.load(f)
        if not sweep:
            continue

        # Load CAZ profile for this model/concept
        caz_path = PAPER_N250 / model_dir_str / f"caz_{concept}.json"
        if not caz_path.exists():
            continue
        with open(caz_path) as f:
            caz_data = json.load(f)

        lm = metrics_from_caz(caz_data)
        prof = find_caz_regions(lm, min_prominence_frac=0.005)
        caz_peak_layers = {r.peak for r in prof.regions}

        for layer_info in sweep["layers"]:
            li = layer_info["layer"]
            red = layer_info["global_sep_reduction"]
            if li in caz_peak_layers:
                all_caz_reds.append(red)
            elif all(abs(li - pk) > 3 for pk in caz_peak_layers):
                all_noncaz_reds.append(red)

if all_caz_reds and all_noncaz_reds:
    mean_caz    = np.mean(all_caz_reds)
    mean_noncaz = np.mean(all_noncaz_reds)
    ratio       = mean_caz / mean_noncaz if mean_noncaz > 0 else float("inf")
    stat, p_val = mannwhitneyu(all_caz_reds, all_noncaz_reds, alternative="greater")
    print(f"Aggregate ablation specificity across {len(available_models)} model dirs:")
    print(f"  CAZ peak layers:    n={len(all_caz_reds):4d}  mean reduction = {mean_caz:.3f}")
    print(f"  Non-CAZ layers:     n={len(all_noncaz_reds):4d}  mean reduction = {mean_noncaz:.3f}")
    print(f"  Ratio:              {ratio:.2f}×")
    print(f"  Mann-Whitney p:     {p_val:.2e}")
    print()
    print("Paper reports: 0.406 / 0.111 = 3.67×  (p = 6.01 × 10⁻³¹)")
else:
    print("No ablation global sweep data available locally.")
    print("Paper result: mean CAZ peak reduction 0.406, non-CAZ 0.111, ratio 3.67×")
    print("Mann-Whitney U = 29,799, p = 6.01 × 10⁻³¹")
    print("This requires running ablation sweeps on GPU.")

## 5. Direction Specificity: Concept Direction vs Random Directions

**Paper claim (§6.1, §6.8):** Ablating the **concept direction** at the CAZ peak
produces a median specificity ratio of **281.6×** over random unit vectors at
the same layer (mean concept reduction 0.416, mean random 0.003; concept wins in
97.8% of pairs).

The behavioral pilot (§6.8) confirms this: random-direction ablation at the CAZ peak
produces near-zero or negative logit-difference suppression (mean rand = −0.013),
while concept-direction ablation suppresses substantially more.

We load the pre-computed `ablation_random_<concept>.json` files from
`~/rosetta_data/paper_n250/<model>/`, which contain both the concept-direction reduction
and 10 random-seed reductions at the same CAZ peak layer.

In [ ]:
def load_random_ablation(model_id: str, concept: str) -> dict | None:
    path = PAPER_N250 / model_dir_name(model_id) / f"ablation_random_{concept}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# Demo: random ablation for all 7 concepts in Pythia-160M
print(f"Direction specificity — {DEMO_MHA.split('/')[-1]}")
print(f"{'Concept':20s}  {'Concept red.':>12}  {'Random mean':>11}  {'Ratio':>8}")

concept_reds, random_means = [], []
for concept in P3_CONCEPTS:
    d = load_random_ablation(DEMO_MHA, concept)
    if d is None:
        print(f"{concept:20s}  (not available)")
        continue
    c_red = d["concept_direction_reduction"]
    r_mean = d["random_mean_reduction"]
    ratio  = c_red / r_mean if r_mean > 1e-6 else float("inf")
    print(f"{concept:20s}  {c_red:>12.4f}  {r_mean:>11.4f}  {ratio:>8.1f}×")
    concept_reds.append(c_red)
    random_means.append(r_mean)

if concept_reds:
    print(f"\nMean concept reduction: {np.mean(concept_reds):.4f}")
    print(f"Mean random reduction:  {np.mean(random_means):.4f}")
else:
    print("\nRandom ablation data not available locally for Pythia-160M.")
    print("Paper result: mean concept 0.416, mean random 0.003, median ratio 281.6×")

In [ ]:
# Aggregate direction specificity from pre-computed random_control summary
rand_ctrl_path = RAND_RESULTS / "random_ablation_control.json"

if rand_ctrl_path.exists():
    with open(rand_ctrl_path) as f:
        rand_ctrl = json.load(f)

    summary = rand_ctrl["summary"]
    overall = summary["overall"]
    cohorts = summary["by_cohort"]

    print("Direction specificity — aggregate results (random_ablation_control.json):")
    print(f"  N pairs:                 {overall['n_pairs']}")
    print(f"  N models:                {overall['n_models']}")
    print(f"  Mean concept reduction:  {overall['mean_concept_red']:.4f}")
    print(f"  Mean random reduction:   {overall['mean_random_red']:.4f}")
    print(f"  Median specificity ratio:{overall['median_ratio']:.1f}×")
    print(f"  Median z-score:          {overall['median_z']:.1f}")
    print(f"  Concept wins all seeds:  {overall['pct_concept_wins_all_seeds']:.1f}%")
    print()
    print("By cohort:")
    for cohort_name, cdata in cohorts.items():
        print(f"  {cohort_name:8s}: n={cdata['n_pairs']:3d}  "
              f"concept={cdata['mean_concept']:.3f}  "
              f"random={cdata['mean_random']:.4f}  "
              f"ratio={cdata['median_ratio']:.1f}×")
    print()
    print("Paper reports: median ratio 281.6× (mean concept 0.416, mean random 0.003)")
    print("Concept direction won all 10 seeds in 97.8% of pairs.")
else:
    print("Pre-computed random ablation summary not found at:")
    print(f"  {rand_ctrl_path}")
    print("Paper result: median specificity ratio 281.6× (mean 139×)")
    print("Concept direction > all 10 random seeds in 97.8% of 182 pairs.")

In [ ]:
# Visualize direction specificity per cohort (reproduces Fig. 6 pattern)
if rand_ctrl_path.exists():
    records = rand_ctrl["records"]
    cohort_data = {}
    for rec in records:
        cohort = rec["cohort"]
        cohort_data.setdefault(cohort, {"concept": [], "random_seeds": []})
        cohort_data[cohort]["concept"].append(rec["concept_red"])
        cohort_data[cohort]["random_seeds"].extend(rec["seed_reds"])

    fig, axes = plt.subplots(1, len(cohort_data), figsize=(4 * len(cohort_data), 5),
                              sharey=True)
    fig.patch.set_facecolor("white")
    fig.suptitle("Direction specificity — concept vs random ablation at CAZ peak layer",
                 fontsize=12, fontweight="bold")

    for ax, (cohort_name, cdata) in zip(axes, cohort_data.items()):
        rand_vals = [v for v in cdata["random_seeds"] if v >= 0]
        if rand_vals:
            parts = ax.violinplot([rand_vals], positions=[0.5], showmedians=True)
            for pc in parts["bodies"]:
                pc.set_facecolor("#9E9E9E")
                pc.set_alpha(0.6)
        ax.scatter([1.2] * len(cdata["concept"]), cdata["concept"],
                   color="#C62828", zorder=5, s=18, alpha=0.7,
                   label="Concept direction")
        ax.set_title(f"{cohort_name}\n(n={len(cdata['concept'])} pairs)",
                     fontsize=10, fontweight="bold")
        ax.set_xticks([0.5, 1.2])
        ax.set_xticklabels(["Random\n(10 seeds)", "Concept\ndirection"], fontsize=8)
        ax.set_ylabel("Separation reduction", fontsize=9)
        apply_theme(ax)

    plt.tight_layout()
    plt.show()
else:
    print("Random ablation control data not available — cannot produce violin plot.")

## 6. Architecture-Conditioned Ablation: MHA vs GQA

**Paper claim (§6.4, Table 12):** Using the GEM peak protocol (ablation at handoff
layer = CAZ end + 1), the MHA+GELU cohort (Pythia, GPT-2, OPT, Phi-2) shows mean
separation reduction of **0.420** while the GQA+SwiGLU cohort (Qwen, Llama, Mistral)
shows **0.261** — a 1.6× gap robust across all models in both cohorts.

This is the central architectural finding: MHA models encode concepts redundantly
across layers (peak ablation terminates one link in the chain); GQA models encode
more sparsely (downstream layers re-derive the concept from other routes).

We load the pre-computed `ablation_gem_<concept>.json` files which were generated
by the GEM protocol on all 26 base models.

In [ ]:
# MHA vs GQA cohort ablation from pre-computed GEM ablation files

# Architecture cohort mapping (base models)
MHA_FAMILIES = {"pythia", "gpt", "gpt2", "opt", "phi"}
GQA_FAMILIES = {"qwen", "llama", "mistral"}
GEMMA_FAMILIES = {"gemma"}

def cohort_of(model_dir_str: str) -> str:
    s = model_dir_str.lower()
    if any(f in s for f in MHA_FAMILIES):
        return "MHA"
    if any(f in s for f in GQA_FAMILIES):
        return "GQA"
    if "gemma" in s:
        return "Gemma"
    return "Unknown"

cohort_reductions = {"MHA": [], "GQA": [], "Gemma": []}

for model_dir_str in available_models:
    cohort = cohort_of(model_dir_str)
    if cohort not in cohort_reductions:
        continue
    # Skip instruct variants
    if any(x in model_dir_str.lower() for x in ["instruct", "it_", "_it"]):
        continue
    for concept in P3_CONCEPTS:
        path = PAPER_N250 / model_dir_str / f"ablation_gem_{concept}.json"
        if not path.exists():
            continue
        with open(path) as f:
            d = json.load(f)
        # GEM ablation JSON: look for the separation reduction
        red = d.get("sep_reduction") or d.get("separation_reduction") or d.get("global_sep_reduction")
        if red is None and "ablation_results" in d:
            red = d["ablation_results"].get("sep_reduction")
        if red is not None:
            cohort_reductions[cohort].append(float(red))

print("GEM peak ablation by cohort (base models only):")
for cohort, reds in cohort_reductions.items():
    if reds:
        n = len(reds)
        mu = np.mean(reds)
        se = np.std(reds, ddof=1) / np.sqrt(n)
        print(f"  {cohort:6s}  n={n:3d}  mean = {mu:.3f} ± {se:.3f} (SEM)")
    else:
        print(f"  {cohort:6s}  (no pre-computed GEM ablation data found)")

print()
print("Paper reports (Table 12):")
print("  MHA    n=112  mean = 0.420")
print("  GQA    n=49   mean = 0.261")
print("  Gemma  n=14   mean = 0.168 (at Fisher peak; ~1.0 at final global layer)")

In [ ]:
# Bar chart: MHA vs GQA cohort ablation comparison
# Use paper numbers if local data is unavailable.

paper_means = {"MHA": 0.420, "GQA": 0.261, "Gemma\n(peak)": 0.168, "Gemma\n(global)": 0.985}
paper_ns    = {"MHA": 112, "GQA": 49, "Gemma\n(peak)": 14, "Gemma\n(global)": 14}
paper_colors= {"MHA": "#1565C0", "GQA": "#E65100", "Gemma\n(peak)": "#2E7D32",
               "Gemma\n(global)": "#7B1FA2"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor("white")

# Left: local data (if available)
ax = axes[0]
local_cohorts = {k: v for k, v in cohort_reductions.items() if v}
if local_cohorts:
    keys   = list(local_cohorts.keys())
    means  = [np.mean(local_cohorts[k]) for k in keys]
    sems   = [np.std(local_cohorts[k], ddof=1)/np.sqrt(len(local_cohorts[k])) for k in keys]
    colors = ["#1565C0", "#E65100", "#2E7D32"][:len(keys)]
    ax.bar(keys, means, color=colors, alpha=0.8, edgecolor="white")
    ax.errorbar(keys, means, yerr=sems, fmt="none", color="black", capsize=5)
    ax.set_title("Local pre-computed data", fontsize=10, fontweight="bold")
else:
    ax.text(0.5, 0.5, "No local GEM ablation data", ha="center", va="center",
            transform=ax.transAxes, fontsize=11, color="#9E9E9E")
ax.set_ylabel("Mean separation reduction", fontsize=10)
ax.set_ylim(0, 1.1)
apply_theme(ax)

# Right: paper numbers
ax = axes[1]
keys = list(paper_means.keys())
ax.bar(keys, [paper_means[k] for k in keys],
       color=[paper_colors[k] for k in keys], alpha=0.8, edgecolor="white")
for k in keys:
    ax.text(keys.index(k), paper_means[k] + 0.02,
            f"n={paper_ns[k]}", ha="center", fontsize=8)
ax.set_title("Paper reported values (Table 12)", fontsize=10, fontweight="bold")
ax.set_ylabel("Mean separation reduction", fontsize=10)
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color="#9E9E9E", lw=1, ls="--", label="Full removal")
apply_theme(ax)

fig.suptitle("Architecture-conditioned ablation: MHA vs GQA vs Gemma",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Gemma-2: ablation at Fisher peak ≈ 0.17; ablation at final global layer ≈ 1.0")
print("Functional allocation in Gemma-2 is at the last global attention layer, not the Fisher peak.")

## 7. Gentle CAZes Are Ablation-Sensitive

**Paper claim (§6.1, Table 10):** Despite having CAZ scores < 0.05, gentle CAZes
produce >20% concept suppression in 93% of ablation tests — compared to 100%
for major CAZes and 20.5% for non-CAZ layers. Score predicts geometric
salience, not causal importance.

| Category | Score | Mean self-retained | Beats non-CAZ (>20% threshold) |
|----------|-------|-------------------|-------------------------------|
| Major | > 0.5 | 30.7% | 100% |
| Strong | 0.2–0.5 | 39.8% | 95% |
| Moderate | 0.05–0.2 | 45.6% | 93% |
| Gentle | < 0.05 | 48.7% | 93% |

"Self-retained" = fraction of pre-ablation separation remaining after ablation
(30.7% retained = 69.3% suppressed). Lower = more concept suppression.

We load individual `ablation_<concept>.json` files to demonstrate the
score vs. suppression relationship.

In [ ]:
def load_ablation(model_id: str, concept: str) -> dict | None:
    """Load the canonical ablation JSON for a model/concept."""
    path = PAPER_N250 / model_dir_name(model_id) / f"ablation_{concept}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# Gather caz_score vs self_retained_pct across available models
score_vs_retained = []  # [(caz_score, self_retained_pct, concept, model)]

for model_dir_str in available_models:
    if any(x in model_dir_str.lower() for x in ["instruct", "_it"]):
        continue
    cohort = cohort_of(model_dir_str)
    for concept in P3_CONCEPTS:
        abl_path = PAPER_N250 / model_dir_str / f"ablation_{concept}.json"
        caz_path = PAPER_N250 / model_dir_str / f"caz_{concept}.json"
        if not (abl_path.exists() and caz_path.exists()):
            continue
        with open(abl_path) as f:
            abl = json.load(f)
        with open(caz_path) as f:
            caz_d = json.load(f)

        # Get self_retained and caz_peak from ablation file
        self_ret = abl.get("self_retained_pct") or abl.get("self_retained")
        if self_ret is None:
            # Try 'layers' with CAZ peak
            if "layers" in abl and "caz_peak" in abl:
                pk = abl["caz_peak"]
                for l in abl["layers"]:
                    if l["layer"] == pk:
                        self_ret = l.get("self_retained_pct") or l.get("ablated_final_sep", 0) / max(abl.get("baseline_final_sep", 1), 1e-6)
                        break
        if self_ret is None:
            continue

        # Get dominant caz_score
        lm = metrics_from_caz(caz_d)
        prof = find_caz_regions(lm, min_prominence_frac=0.005)
        dom_score = prof.dominant.caz_score

        score_vs_retained.append((dom_score, float(self_ret) * 100, concept, model_dir_str, cohort))

print(f"Gathered {len(score_vs_retained)} (model, concept) ablation pairs")

In [ ]:
if score_vs_retained:
    scores   = np.array([r[0] for r in score_vs_retained])
    retained = np.array([r[1] for r in score_vs_retained])
    cohorts  = [r[4] for r in score_vs_retained]
    cohort_colors = {"MHA": "#1565C0", "GQA": "#E65100", "Gemma": "#2E7D32", "Unknown": "#9E9E9E"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("white")

    # Left: scatter caz_score vs self_retained
    ax = axes[0]
    for c_name, c_color in cohort_colors.items():
        idx = [i for i, c in enumerate(cohorts) if c == c_name]
        if idx:
            ax.scatter(scores[idx], retained[idx], s=15, alpha=0.5,
                       color=c_color, label=c_name)
    ax.axhline(50, color="#9E9E9E", lw=1, ls="--", label="50% retained")
    ax.set_xlabel("CAZ score (geometric prominence)", fontsize=10)
    ax.set_ylabel("Self-retained separation (%) — lower = more suppression", fontsize=10)
    ax.set_title("CAZ score vs ablation suppression\nScore predicts salience, not causal importance",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    apply_theme(ax)

    # Right: mean retained by score category
    ax = axes[1]
    cat_order = ["gentle", "moderate", "strong", "major"]
    cat_means  = []
    cat_labels = []
    for cat in cat_order:
        if cat == "gentle":  mask = scores < 0.05
        elif cat == "moderate": mask = (scores >= 0.05) & (scores < 0.2)
        elif cat == "strong":   mask = (scores >= 0.2) & (scores < 0.5)
        else:                   mask = scores >= 0.5
        vals = retained[mask]
        if len(vals) > 0:
            cat_means.append(float(vals.mean()))
            cat_labels.append(f"{cat}\n(n={len(vals)})")
    ax.bar(cat_labels, cat_means, color=["#1565C0", "#2E7D32", "#E65100", "#C62828"],
           alpha=0.8, edgecolor="white")
    ax.set_ylabel("Mean self-retained (%)", fontsize=10)
    ax.set_title("Mean ablation retention by CAZ score category",
                 fontsize=11, fontweight="bold")
    ax.set_ylim(0, 80)
    apply_theme(ax)

    plt.tight_layout()
    plt.show()

    # Suppress ratio by category
    print("Mean self-retained by score category (lower = more concept suppressed):")
    for cat in cat_order:
        if cat == "gentle":  mask = scores < 0.05
        elif cat == "moderate": mask = (scores >= 0.05) & (scores < 0.2)
        elif cat == "strong":   mask = (scores >= 0.2) & (scores < 0.5)
        else:                   mask = scores >= 0.5
        vals = retained[mask]
        if len(vals):
            print(f"  {cat:10s}: n={len(vals):3d}  mean retained={vals.mean():.1f}%  "
                  f"(suppressed={100-vals.mean():.1f}%)")
    print()
    print("Paper Table 10 reports (from 467 ablation sweeps):")
    print("  major:    30.7% retained  → 69.3% suppressed")
    print("  strong:   39.8% retained  → 60.2% suppressed")
    print("  moderate: 45.6% retained  → 54.4% suppressed")
    print("  gentle:   48.7% retained  → 51.3% suppressed")
else:
    print("No ablation data found locally. Cannot compute score vs suppression chart.")
    print("Paper Table 10 (467 ablation sweeps):")
    print("  major:   30.7% retained   strong:  39.8%")
    print("  moderate: 45.6% retained  gentle: 48.7%")
    print("All categories: 93%+ beat the non-CAZ baseline (11.1% suppression)")

## 8. Multimodal Allocation

**Paper claim (§4.2):** Concepts assemble at multiple depths. Across 26 models and
7 concepts, the **mean is 3.4 CAZes per concept per model** under scored detection.
Credibility is the most multimodal (73% of models show 2+ peaks). Every concept
shows multimodality in at least some models.

Sub-representations at different depths are geometrically distinct: within-model
cosine similarity between shallow and deep peak directions averages **0.379** for
credibility (range 0.14–0.59) — well above random baseline (~0.02) but far below
identity. The same concept is represented by geometrically distinct directions at
different depths.

We demonstrate the per-peak cosine measurement that underpins this finding.

In [ ]:
# Demonstrate inter-peak cosine computation for credibility in Pythia-160M
# (replicates the pattern from §3.3 and Table 5)

def inter_peak_cosines(data: dict, min_prominence: float = 0.005) -> list[dict]:
    """Compute cosine similarity between adjacent peak directions."""
    lm = metrics_from_caz(data)
    n_layers = data["n_layers"]
    prof = find_caz_regions(lm, min_prominence_frac=min_prominence)

    results = []
    metrics = data["layer_data"]["metrics"]
    for i in range(len(prof.regions) - 1):
        r1, r2 = prof.regions[i], prof.regions[i + 1]
        v1 = np.array(metrics[r1.peak]["dom_vector"])
        v2 = np.array(metrics[r2.peak]["dom_vector"])
        cos = float(abs(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12)))
        results.append({
            "shallow_peak_pct": r1.depth_pct,
            "deep_peak_pct": r2.depth_pct,
            "cosine": cos,
        })
    return results

print("Inter-peak cosine similarity — Pythia-160M (12L MHA)")
print("Paper Table 5 reference values (cross-model means):")
print("  credibility: 0.379 · causation: 0.212 · temporal_order: 0.156")
print()

for concept in P3_CONCEPTS:
    data = load_caz(DEMO_MHA, concept)
    pairs = inter_peak_cosines(data)
    n_layers = data["n_layers"]
    hdim = data.get("hidden_dim", 0)
    chance = 1.0 / np.sqrt(hdim) if hdim > 0 else 0.0
    if pairs:
        cos_str = ", ".join(f"{p['cosine']:.3f}" for p in pairs)
        print(f"  {concept:20s}  {len(pairs)+1} peaks  cosines: {cos_str}")
    else:
        print(f"  {concept:20s}  1 peak   (unimodal)")

print(f"\nChance baseline ({hdim}-dim): {chance:.4f}")
print("Values well above chance but well below 1.0 → same concept, distinct sub-representations.")

In [ ]:
# Full topology chart: all 7 concepts in Pythia-160M (reproduces Fig. 1 approach)
concept_order_local = sorted(
    P3_CONCEPTS,
    key=lambda c: caz_mha[c]["layer_data"]["peak_depth_pct"]
)

bar_h    = 0.20
row_gap  = 0.08
row_step = bar_h + row_gap
n_rows   = len(concept_order_local)

fig, ax = plt.subplots(figsize=(12, n_rows * row_step + 0.6))
fig.patch.set_facecolor("white")

_nl = caz_mha[P3_CONCEPTS[0]]["n_layers"]
for ci, concept in enumerate(concept_order_local):
    y = ci * row_step
    data = caz_mha[concept]
    lm   = metrics_from_caz(data)
    prof = find_caz_regions(lm, min_prominence_frac=0.005)
    color = concept_color(concept)
    for i, region in enumerate(prof.regions):
        s_pct = region.start / _nl * 100
        e_pct = region.end   / _nl * 100
        alpha = max(0.85 - i * 0.22, 0.25)
        ax.barh(y, e_pct - s_pct, left=s_pct, height=bar_h,
                color=color, alpha=alpha, zorder=2)
        ax.plot(region.depth_pct, y, "o", color=color, ms=5, zorder=3,
                markeredgecolor="white", markeredgewidth=0.5)

ax.set_yticks([ci * row_step for ci in range(n_rows)])
ax.set_yticklabels([c.replace("_", " ") for c in concept_order_local], fontsize=10)
ax.set_xlim(0, 100)
ax.set_xlabel("Depth (% of layers)", fontsize=11)
ax.set_title(
    f"Concept topology — Pythia-160M ({_nl}L MHA)\n"
    "Bar = zone extent · dot = peak · faded = secondary zones (scored 0.5% floor)",
    fontsize=12, fontweight="bold",
)
ax.set_ylim(-bar_h, (n_rows - 1) * row_step + bar_h)
apply_theme(ax)
plt.tight_layout()
plt.show()

## 9. CAZ Dependency Structure: Forward-Only Information Flow

**Paper claim (§6.2):** Directional ablation of 999 CAZ pairs across 25 base models:
- **58% independent**: ablating one CAZ has no effect on the other
- **42% forward-dependent**: shallow CAZ feeds into deeper one
- **0% backward or coupled**: not a single backward dependency found

The 0% backward rate is expected from the residual stream architecture (already-computed
layers cannot be affected by downstream ablation). The 58/42 forward split indicates
that most CAZes operate on independent geometric directions; the dependent minority
reveals hierarchical computation: shared shallow allocations feeding downstream gentle
refinement.

This finding is based on the multimodal ablation sweep — running directional ablation
at each CAZ and measuring its effect on all other CAZes in the same concept. The
pre-computed results are available in the ablation JSON files.

In [ ]:
# Demonstrate the concept: for a multimodal concept, check if ablating the
# upstream (shallower) peak suppresses the downstream (deeper) peak's separation.

# Load the ablation file for a concept with multiple zones in Pythia-160M
concept = "credibility"  # most multimodal in our demo model
abl = load_ablation(DEMO_MHA, concept)
caz_d = caz_mha[concept]
lm = metrics_from_caz(caz_d)
prof = find_caz_regions(lm, min_prominence_frac=0.005)

n_layers = caz_d["n_layers"]
print(f"Credibility in Pythia-160M: {prof.n_regions} zones")
for zi, r in enumerate(prof.regions):
    print(f"  Z{zi+1}: peak layer {r.peak} ({r.depth_pct:.0f}%)  score={r.caz_score:.3f}  ({score_category(r.caz_score)})")

print()
print("Dependency structure (paper §6.2):")
print("  58% of all cross-zone pairs: independent (ablating Z1 has no effect on Z2)")
print("  42% forward-dependent: ablating shallow zone partially suppresses deeper zone")
print("   0% backward: no downstream ablation affects already-computed shallow layers")
print()

if abl and "layers" in abl:
    # Show ablation reduction at each layer relative to baseline
    baseline = abl.get("baseline_final_sep", None)
    peak_layers = {r.peak for r in prof.regions}
    print(f"Global ablation sweep at each layer (baseline sep = {baseline:.4f}):")
    for l in abl["layers"]:
        li = l["layer"]
        red = l.get("global_sep_reduction", None)
        marker = " ◄ CAZ peak" if li in peak_layers else ""
        if red is not None:
            print(f"  Layer {li:2d} ({100*li/n_layers:.0f}%): reduction = {red:.4f}{marker}")
else:
    print("Ablation sweep data not available locally.")
    print("Paper: single-CAZ ablation confirms each zone is independently functional.")

## 10. Width and Abstraction (P3)

**Paper claim (§5.3):** More abstract concepts have wider CAZes.
Excluding credibility (bimodal, high width variance), the correlation between
abstraction level and CAZ width is **r = 0.294, p = 0.003** (n = 132 sweeps).

Pairwise: certainty wider than negation by 18.5% (p = 0.002); moral valence
wider than causation by 18.3% (p = 0.007). This is a **partially supported**
prediction — the effect exists for compositionally assembled concepts but
breaks for surface-cue-dominant concepts.

Concept abstraction ranking used in the paper (researcher-assigned):
`negation < causation < temporal_order < sentiment < certainty < moral_valence`

In [ ]:
# Compute mean dominant CAZ width by concept from scored_analysis.csv

# Abstraction ordering (negation = most concrete; moral_valence = most abstract)
ABSTRACTION_RANK = {
    "negation": 1, "causation": 2, "temporal_order": 3,
    "sentiment": 4, "certainty": 5, "moral_valence": 6,
    # credibility excluded from P3 width test (bimodal)
}

concept_widths = {c: [] for c in ABSTRACTION_RANK}

with open(scored_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["concept"] not in ABSTRACTION_RANK:
            continue
        if row["is_dominant"] != "True":
            continue
        concept_widths[row["concept"]].append(float(row["width_pct"]))

print("Mean dominant CAZ width (%) by concept — abstraction order:")
print(f"{'Concept':20s}  {'Abstraction':>12}  {'N':>5}  {'Mean width':>10}  {'Std':>8}")
rank_vals  = []
width_vals = []
for concept in sorted(ABSTRACTION_RANK, key=ABSTRACTION_RANK.get):
    w = concept_widths[concept]
    if w:
        mean_w = np.mean(w)
        std_w  = np.std(w, ddof=1)
        print(f"{concept:20s}  {ABSTRACTION_RANK[concept]:>12}  {len(w):>5}  {mean_w:>10.2f}%  {std_w:>8.2f}%")
        rank_vals.append(ABSTRACTION_RANK[concept])
        width_vals.append(mean_w)

if rank_vals:
    from scipy.stats import pearsonr
    r, p = pearsonr(rank_vals, width_vals)
    print(f"\nPearson r (abstraction rank vs mean width) = {r:.3f}  (p = {p:.4f})")
    print("Paper reports: r = 0.294, p = 0.003 (n=132 sweeps, credibility excluded)")

## Summary

| Section | Paper claim | Status in this notebook |
|---------|------------|------------------------|
| 1. Concept ordering | Median τ = 0.54, 87% positive, p < 0.001 | Reproduced from scored_analysis.csv |
| 2. Phi-2 inversion | τ = −0.25, reversed ordering | Direct computation on model data |
| 3. Scored detection | 201 → 623 CAZes (3.1×) | Counts from scored_analysis.csv |
| 4. Suppression ratio | 3.67× at CAZ vs non-CAZ layers | From ablation_global_sweep files |
| 5. Direction specificity | Median ratio 281.6× concept vs random | From random_ablation_control.json |
| 6. MHA vs GQA ablation | 0.420 vs 0.261 (GEM peak protocol) | From ablation_gem files |
| 7. Gentle CAZes | 93% beat non-CAZ threshold (all categories) | From ablation files |
| 8. Multimodal allocation | Mean 3.4 CAZes/concept/model, cos 0.379 | Direct computation |
| 9. Forward-only flow | 0% backward, 42% forward dependencies | Described; ablation file loading shown |
| 10. Width and abstraction | r = 0.294, p = 0.003 | Computed from scored_analysis.csv |

---

### Papers and resources

| | |
|-|-|
| Paper 1 — CAZ Framework | Henry (2026a) |
| Paper 2 — GEM | Henry (2026b) |
| Paper 3 — CAZ Validation | Henry (2026c) |
| Paper 4 — PRH | Henry (2026d) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |
| Rosetta Activations dataset | [HuggingFace](https://huggingface.co/datasets/james-ra-henry/Rosetta-Activations) |
| Concept pairs dataset | [Rosetta_Concept_Pairs](https://github.com/jamesrahenry/Rosetta_Concept_Pairs) |

### Data requirements

| Section | Data needed | Source |
|---------|------------|--------|
| 1–3, 8, 10 | `paper_n250/*/caz_*.json` + `results/CAZ_Validation/scored_analysis.csv` | HF dataset or local |
| 4 | `paper_n250/*/ablation_global_sweep_*.json` | Pre-computed (GPU) |
| 5 | `results/random_control/random_ablation_control.json` | Pre-computed (GPU) |
| 6 | `paper_n250/*/ablation_gem_*.json` | Pre-computed (GPU) |
| 7 | `paper_n250/*/ablation_*.json` | Pre-computed (GPU) |
| 9 | `paper_n250/*/ablation_*.json` | Pre-computed (GPU) |